# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-07: HR-head training DataLoader

### What this is
A PyTorch DataLoader over the three preprocessed Phase-3 datasets (MCD + DLCN + UBFC-rPPG, ~2000 recordings on the unified 72x72 grouped schema) for training the HR head. Kaggle-ready: it reads the consolidated per-dataset HDF5 files on-demand, opens one file handle per worker (the standard h5 + multiprocessing pattern), and yields fixed-length clips.

### What each clip provides
- frames: standardized face-crop clip, shape [3, T, 72, 72] (channels, time, H, W — PhysFormer/PhysNet convention)
- bvp: the frame-synced pulse waveform, standardized, shape [T] (the reconstruction target; HR is derived from it)
- hr: the clip's heart rate in bpm (spectral peak of the bvp) — for balancing and eval
- sqi: cardiac_sqi, to be used as a soft per-sample loss weight
- speed: the temporal-rescale factor applied (1.0 = none)
- dataset, subject: provenance

### Design choices baked in
- **Subject-wise split** (namespaced by dataset) so no subject is in both train and val.
- **Temporal-rescale augmentation** : resample a T*s-frame window to T frames (and the bvp identically) to synthesise higher/lower apparent HR, flattening the HR distribution and countering the ~88 bpm shrinkage. Applied to a fraction of training clips.
- **SQI soft-weighting** exposed per sample (MCD labels are noisier than DLCN/UBFC).
- Horizontal-flip augmentation (pulse-invariant).
- Raw frames were preserved in preprocessing; normalization (per-clip standardization) happens here in the loader.


In [10]:
import time
from pathlib import Path
from collections import Counter
import numpy as np
import h5py
import torch
from torch.utils.data import Dataset, DataLoader

# Data location
DATA_DIR = Path('/kaggle/input/datasets/cezarytubacki/phase3-hr-train-dataset')

# Auto-discover H5 files in the dataset
H5_FILES = sorted([f.name for f in DATA_DIR.glob('*.h5')])

# Config parameters
CLIP_LEN = 160  # frames per clip (~5 s at 30 fps)
IMG_SIZE = 72
BATCH_SIZE = 16
NUM_WORKERS = 2
VAL_FRAC = 0.2
SEED = 42

# Temporal-rescale augmentation (HR-distribution flattening)
AUG_PROB = 0.5
SPEED_MIN, SPEED_MAX = 0.7, 2.0  # <1 slows (lower HR), >1 speeds (higher HR)
SQI_FLOOR = 0.05

print('DATA_DIR:', DATA_DIR)
print('H5 files found:', H5_FILES)

DATA_DIR: /kaggle/input/datasets/cezarytubacki/phase3-hr-train-dataset
H5 files found: ['dlcn_phase3.h5', 'mcd_phase3.h5', 'ubfc_rppg_phase3.h5']


In [11]:
def build_index(data_dir, h5_files, sqi_floor=SQI_FLOOR, clip_len=CLIP_LEN):
    '''Scans the dataset HDF5 files (attributes only) and returns a list of usable-recording entries.'''
    index = []
    for fn in h5_files:
        path = Path(data_dir) / fn
        if not path.exists():
            print('missing', path)
            continue
        with h5py.File(path, 'r') as f:
            for name in f.keys():
                a = f[name].attrs
                if not bool(a.get('usable', True)):
                    continue
                if float(a.get('cardiac_sqi', 0.0)) < sqi_floor:
                    continue
                n = int(a['n_frames'])
                if n < clip_len:
                    continue
                index.append(dict(file=fn, group=name,
                                  dataset=str(a.get('dataset', fn)),
                                  subject=str(a.get('subject_id', name)),
                                  n_frames=n, fps=float(a.get('fps', 30.0)),
                                  sqi=float(a.get('cardiac_sqi', 0.0)),
                                  state=str(a.get('state', 'unknown'))))
    return index


def subject_split(index, val_frac=VAL_FRAC, seed=SEED):
    '''Subject-wise train/val split (no subject in both), namespaced by dataset.'''
    import random
    subjects = sorted({r['dataset'] + '/' + r['subject'] for r in index})
    rng = random.Random(seed)
    rng.shuffle(subjects)
    n_val = max(1, int(round(len(subjects) * val_frac)))
    val_subjects = set(subjects[:n_val])
    train = [r for r in index if (r['dataset'] + '/' + r['subject']) not in val_subjects]
    val = [r for r in index if (r['dataset'] + '/' + r['subject']) in val_subjects]
    return train, val, val_subjects

In [12]:
def compute_hr(bvp, fps, low=0.7, high=3.5):
    '''Returns the cardiac-band spectral-peak HR (bpm) of a bvp clip, or nan.'''
    x = np.asarray(bvp, dtype=np.float64)
    x = x - x.mean()
    if x.std() < 1e-8:
        return float('nan')
    p = np.abs(np.fft.rfft(x * np.hanning(len(x)))) ** 2
    fr = np.fft.rfftfreq(len(x), d=1.0 / fps)
    b = (fr >= low) & (fr <= high)
    if not b.any() or p[b].sum() <= 0:
        return float('nan')
    return float(fr[b][int(np.argmax(p[b]))] * 60.0)


class RPPGClipDataset(Dataset):
    '''Yields fixed-length standardized face-crop clips + frame-synced bvp label from the Phase-3 HDF5 files.'''
    def __init__(self, index, data_dir, clip_len=CLIP_LEN, augment=False,
                 aug_prob=AUG_PROB, speed_range=(SPEED_MIN, SPEED_MAX), seed=SEED):
        self.index = index
        self.data_dir = Path(data_dir)
        self.clip_len = clip_len
        self.augment = augment
        self.aug_prob = aug_prob
        self.speed_range = speed_range
        self.seed = seed
        self._files = {}
        self._rng = np.random.default_rng(seed)

    def _h5(self, fn):
        h = self._files.get(fn)
        if h is None:
            h = h5py.File(self.data_dir / fn, 'r')
            self._files[fn] = h
        return h

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        rec = self.index[i]
        g = self._h5(rec['file'])[rec['group']]
        n, T = rec['n_frames'], self.clip_len
        if self.augment and self._rng.random() < self.aug_prob:
            src_len = int(round(T * float(self._rng.uniform(*self.speed_range))))
            src_len = max(T, min(src_len, n))
        else:
            src_len = T
        s = src_len / T
        max_start = n - src_len
        start = int(self._rng.integers(0, max_start + 1)) if (self.augment and max_start > 0) else max_start // 2
        frames_src = g['frames'][start:start + src_len].astype(np.float32)
        bvp_src = g['bvp'][start:start + src_len].astype(np.float32)
        pos = np.linspace(0, src_len - 1, T)
        frames = frames_src[np.rint(pos).astype(int)]
        bvp = np.interp(pos, np.arange(src_len), bvp_src)
        hr_bpm = compute_hr(bvp, rec['fps'])
        if self.augment and self._rng.random() < 0.5:
            frames = frames[:, :, ::-1, :]
        m = frames.mean(axis=(0, 1, 2), keepdims=True)
        sd = frames.std(axis=(0, 1, 2), keepdims=True) + 1e-6
        frames = np.ascontiguousarray((frames - m) / sd)
        frames = torch.from_numpy(frames).permute(3, 0, 1, 2).float()
        bvp = (bvp - bvp.mean()) / (bvp.std() + 1e-6)
        bvp = torch.from_numpy(bvp.astype(np.float32))
        return dict(frames=frames, bvp=bvp,
                    hr=torch.tensor(hr_bpm, dtype=torch.float32),
                    sqi=torch.tensor(rec['sqi'], dtype=torch.float32),
                    speed=torch.tensor(s, dtype=torch.float32),
                    dataset=rec['dataset'], subject=rec['subject'])


def worker_init_fn(worker_id):
    '''Gives each DataLoader worker its own RNG and fresh HDF5 handles.'''
    info = torch.utils.data.get_worker_info()
    ds = info.dataset
    ds._files = {}
    ds._rng = np.random.default_rng(SEED + 1000 * (worker_id + 1) + int(info.seed % 100000))

In [13]:
def make_loader(index, data_dir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                clip_len=CLIP_LEN, augment=False):
    '''Builds a DataLoader; training loaders shuffle + augment + drop_last, val loaders do not.'''
    ds = RPPGClipDataset(index, data_dir, clip_len=clip_len, augment=augment)
    return DataLoader(ds, batch_size=batch_size, shuffle=augment, num_workers=num_workers,
                      worker_init_fn=worker_init_fn, pin_memory=True, drop_last=augment,
                      persistent_workers=(num_workers > 0))

### Sanity test
Builds the index, makes the subject-wise split, pulls one training batch (shapes + first-batch load time as a Kaggle I/O check), and confirms the temporal augmentation actually widens the HR distribution above 85 bpm.

In [14]:
index = build_index(DATA_DIR, H5_FILES)
print('usable recordings:', len(index), '| by dataset:', dict(Counter(r['dataset'] for r in index)))

train_idx, val_idx, val_subj = subject_split(index)
print('train recs:', len(train_idx), '| val recs:', len(val_idx), '| val subjects:', len(val_subj))

train_loader = make_loader(train_idx, DATA_DIR, augment=True)
val_loader = make_loader(val_idx, DATA_DIR, augment=False)

t0 = time.time()
batch = next(iter(train_loader))
print('first batch loaded in', round(time.time() - t0, 1), 's')
print('frames', tuple(batch['frames'].shape), batch['frames'].dtype, '| bvp', tuple(batch['bvp'].shape))
print('hr', [round(float(x)) for x in batch['hr'][:8]])
print('sqi', [round(float(x), 2) for x in batch['sqi'][:8]])
print('speed', [round(float(x), 2) for x in batch['speed'][:8]])


def hr_spread(idx, augment, n=200):
    ds = RPPGClipDataset(idx, DATA_DIR, augment=augment)
    pick = np.random.default_rng(0).integers(0, len(ds), n)
    hrs = np.array([float(ds[int(k)]['hr']) for k in pick])
    return hrs[np.isfinite(hrs)]

no_aug = hr_spread(train_idx, augment=False)
aug = hr_spread(train_idx, augment=True)
print('HR no-aug: min', round(no_aug.min()), 'med', round(np.median(no_aug)), 'max', round(no_aug.max()),
      '| >85bpm', round((no_aug > 85).mean() * 100), '%')
print('HR aug   : min', round(aug.min()), 'med', round(np.median(aug)), 'max', round(aug.max()),
      '| >85bpm', round((aug > 85).mean() * 100), '%')

usable recordings: 2010 | by dataset: {'DLCN': 777, 'MCD': 1191, 'UBFC-rPPG': 42}
train recs: 1585 | val recs: 425 | val subjects: 148
first batch loaded in 5.3 s
frames (16, 3, 160, 72, 72) torch.float32 | bvp (16, 160)
hr [70, 94, 90, 82, 106, 68, 79, 94]
sqi [0.26, 0.1, 0.29, 0.23, 0.19, 0.58, 0.54, 0.23]
speed [1.17, 1.05, 1.19, 1.0, 1.32, 1.0, 1.0, 1.0]
HR no-aug: min 47 med 90 max 135 | >85bpm 57 %
HR aug   : min 45 med 94 max 202 | >85bpm 70 %
